In [1]:
# some variables/calculations apply to all the subsets of data (all figures).
# to apply these, run '%run All_notebooks.ipynb' in the first cell of a notebook.
# Note that several variable names will be pre-defined, including:
# lcl(), data, tthr, sthr, sample_dictionary, transfected, a, bins

In [2]:
# import packages
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.widgets import PolygonSelector
from matplotlib.colors import LogNorm
from matplotlib.ticker import FixedLocator
# import matplotlib
# from matplotlib.widgets import PolygonSelector
# from matplotlib.colors import LogNorm
import seaborn as sns
import xarray as xr
import flowkit as fk
import glob
import os
import math
from scipy.stats import linregress, ttest_ind, f_oneway, pearsonr

# configure interactive plots
# %matplotlib ipympl

In [3]:
# set parameters for plots/visualization
SMALL_SIZE = 7
MEDIUM_SIZE = 7
BIGGER_SIZE = 7

plt.rcParams['svg.fonttype'] = 'none'    # makes text editable in exported .svg files
plt.rcParams['font.sans-serif'] = 'Arial'# set font for all plots to Arial

plt.rc('font', family='Arial')
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)    # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [4]:
n = '20241114'

In [5]:
# load in data
samplenames = [str.join('_', file.split('_')[3:]).split('.')[0] for file in glob.glob('./'+n+'/*.fcs')]
samplefiles = {str.join('_', file.split('_')[3:]).split('.')[0] : file for file in glob.glob('./'+n+'/*.fcs')}

# create flowkit Sample objects
all_experiments = { d : fk.Sample(samplefiles[d], subsample = 2000000) for d in samplefiles }

# reformat as dataframes
data = { smpl : all_experiments[smpl].as_dataframe(source = 'raw') for smpl in all_experiments }
for j in np.arange(len(samplenames)):
    data[samplenames[j]].columns = data[samplenames[j]].columns.droplevel(-1)

In [6]:
# Drawing gates (function written by Kira Buttrey)
def draw_gates(sample, x_channel, y_channel, x_scale='linear', y_scale='linear', title='Gate'):
    # Create a figure and a 2x2 grid of subplots
    fig = plt.figure(figsize=(8, 8))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 0.2], height_ratios=[0.2, 1], wspace=0.05, hspace=0.05)

    # Create the main scatterplot axes
    ax_main = fig.add_subplot(gs[1, 0])

    # Create the histogram axes, sharing x and y axes with the main axes
    ax_top = fig.add_subplot(gs[0, 0], sharex=ax_main)
    ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)

    # Plot the main scatterplot
    ax_main.scatter(sample.as_dataframe(source='raw')[x_channel], 
                    sample.as_dataframe(source='raw')[y_channel], 
                    s=1, alpha=0.5)

    # Set labels and scales for the main axes
    ax_main.set_xlabel(x_channel)
    ax_main.set_ylabel(y_channel)
    ax_main.set_xscale(x_scale)
    ax_main.set_yscale(y_scale)

    #get x-axis and y-axis limits
    xmin, xmax, ymin, ymax = ax_main.axis()

    # Calculate histogram bins based on the selected scale
    if x_scale == 'log':
        x_bins = np.logspace(np.log10(xmin), np.log10(xmax), num=100)
    else:
        x_bins = np.linspace(xmin, xmax, num=100)

    if y_scale == 'log':
        y_bins = np.logspace(np.log10(ymin), np.log10(ymax), num=100)
    else:
        y_bins = np.linspace(ymin, ymax, num=100)

    # Plot the x-axis histogram
    ax_top.hist(sample.as_dataframe(source='raw')[x_channel], bins=x_bins, alpha=0.5)
    ax_top.set_ylabel('Count')
    ax_top.tick_params(axis='both', which='both', labelbottom=False, labelleft=False, bottom=False, left=False)

    # Plot the y-axis histogram
    ax_right.hist(sample.as_dataframe(source='raw')[y_channel], bins=y_bins, orientation='horizontal', alpha=0.5)
    ax_right.set_xlabel('Count')
    ax_right.tick_params(axis='both', which='both', labelbottom=False, labelleft=False, bottom=False, left=False)

    # Set the title above the top histogram
    ax_top.set_title(title)

    # Function to handle polygon selection
    global selected_verts
    selected_verts = None
    def onselect(verts):
        global selected_verts
        selected_verts = verts
        print("Selected polygon coordinates:")
        print(verts)
        
    # Call PolygonSelector
    poly1 = PolygonSelector(ax_main, onselect)

    # Show the plot
    plt.tight_layout()
    plt.show()
    
    return selected_verts

In [7]:
# drawing Forward/Side scatter Area gate based on control sample
g_strat = fk.GatingStrategy()
Sample_Name = all_experiments['untransfected_unstained']
# gate1_verts = draw_gates(Sample_Name, 'FSC-A', 'SSC-A', x_scale='linear', y_scale='linear', title='Sample 1, Gate 1')

# After running this cell, an interactive figure will pop up on which the user can draw a polygon to create a gate.
# After closing the polygon, closing the figure will store the selected vertices as gate1_verts; they will also be printed below the cell.
# The printed vertex coordinates can be pasted (as below) to define the same gate when running the code again, or to keep gates consistent across experiments.
gate1_verts = [(377321.2597242923, 8338.522810836603), (555328.5473462305, 12273.477623282663), (685169.157141056, 18503.822742988923), (777314.1060277063, 29160.992026697), (840140.2075413315, 41785.63871662811), (815009.7669358815, 48671.80963840871), (779408.3094114938, 53918.41605500346), (752183.6654222562, 54738.198307596394), (708205.3943627186, 48999.72253944589), (565799.564265168, 38670.46615677498), (444335.76800549263, 27357.471070992557), (360567.6326539923, 17520.08403987741), (337531.3954323296, 12273.477623282663)]

In [8]:
# Define gate based on vertices
dim_a = fk.Dimension('FSC-A') 
dim_b = fk.Dimension('SSC-A')
poly_gate = fk.gates.PolygonGate('poly1', dimensions = [dim_a, dim_b], vertices = gate1_verts) 

#add gate to strategy on root
g_strat.add_gate(poly_gate, gate_path=('root',))

In [9]:
# drawing Forward Scatter Height vs Side Scatter Area gate based on control sample to select single cells
# gate2_verts = draw_gates(Sample_Name, 'SSC-A', 'SSC-H', title = 'Sample 1, Gate 2')

# After running this cell, an interactive figure will pop up on which the user can draw a polygon to create a gate.
# After closing the polygon, closing the figure will store the selected vertices as gate1_verts; they will also be printed below the cell.
# The printed vertex coordinates can be pasted (as below) to define the same gate when running the code again, or to keep gates consistent across experiments.
gate2_verts = [(1834.0358882898545, 4991.187852540155), (8336.199863874397, 5247.1074881709355), (30484.195905709246, 13436.535828355922), (59134.35592312865, 29687.4326909105), (63198.20840786898, 35445.62449260307), (59743.93379583969, 40052.17793395713), (51616.228826359016, 39028.499391434), (30687.388529946264, 27895.99524149504)]

In [10]:
# add gate 2 to gating strategy after gate 1
dim_a = fk.Dimension('SSC-A') 
dim_b = fk.Dimension('SSC-H') 

poly_gate2 = fk.gates.PolygonGate('poly2', dimensions = [dim_a, dim_b], vertices = gate2_verts) 

g_strat.add_gate(poly_gate2, gate_path=('root', 'poly1'))

In [11]:
# Saving gated data
df_list = []
channels = ['Event','Time','FSC-A','SSC-A','FSC-H','SSC-H','BL1-H','YL1-H',]
for expt in all_experiments:
    df = all_experiments[expt].as_dataframe(source = 'raw', col_multi_index=False)
    res = g_strat.gate_sample(all_experiments[expt])
    include = res.get_gate_membership('poly1')
    g1 = all_experiments[expt].as_dataframe(source = 'raw', col_multi_index=False)[channels].loc[include]
    res = g_strat.gate_sample(all_experiments[expt])
    include = res.get_gate_membership('poly2')
    g2 = all_experiments[expt].as_dataframe(source = 'raw', col_multi_index=False)[channels].loc[include]
    g2['S'] = [expt]*len(g2)
    g2['E'] = n
    g2['R'] = [0]*len(g2)
    g2.index.set_names('CellID', inplace=True)
    g2 = g2.reset_index().set_index(list('ESR')+['CellID'])
    df_list.append(g2)

In [12]:
n

'20241114'

In [13]:
multidata = pd.concat(df_list).dropna(how = 'any')
multidata.to_csv(n+'.csv', index_label=multidata.index.names)

In [ ]:
# calculating gated fraction
summary = pd.DataFrame(index = ['Event Count', 'g1 fraction', 'g2 fraction of g1', 'g2 fraction of all events'])
channels = ['Event','Time','FSC-A','SSC-A','FSC-H','SSC-H','BL1-H','YL1-H',]
for expt in all_experiments:
    df = all_experiments[expt].as_dataframe(source = 'raw', col_multi_index=False)
    all_denom = len(df)
    res = g_strat.gate_sample(all_experiments[expt])
    include = res.get_gate_membership('poly1')
    g1_num = sum(include)
    g2_denom = g1_num
    g1 = all_experiments[expt].as_dataframe(source = 'raw', col_multi_index=False)[channels].loc[include]
    res = g_strat.gate_sample(all_experiments[expt])
    include = res.get_gate_membership('poly2')
    g2_num = sum(include)
    g2 = all_experiments[expt].as_dataframe(source = 'raw', col_multi_index=False)[channels].loc[include]
    g2['S'] = [expt]*len(g2)
    g2['E'] = [samplefiles[expt].split('_')[0]]*len(g2)
    g2['R'] = [0]*len(g2)
    g2.index.set_names('CellID', inplace=True)
    g2 = g2.reset_index().set_index(list('ESR')+['CellID'])
    summary[expt] = [len(df), g1_num/all_denom, g2_num/g2_denom, g2_num/all_denom]
summary = summary.sort_values(by = 'g1 fraction', axis = 1).T
summary.index.set_names('S', inplace = True)
summary

In [ ]:
fig, axs = plt.subplots(1,4,layout = 'constrained', dpi = 150, sharey = True)
for col, ax in zip(summary.columns, axs):
    sns.barplot(summary, x = col, y = 'S', ax = ax)
    ax.set_title(col)
    ax.set_ylabel(None)
    if col != 'Event Count':
        ax.set_xlim(0,1)
plt.show()

In [ ]:
summary.mean()

In [ ]:
summary.min()

In [ ]:
summary.max()

In [ ]:
summary.to_csv('./'+n+'/'+n+'_gating_summary.csv')